# Скок-поскок по "Ситону"

В этой заметке пробегусь по ["основам языка \[Cython\]"](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html). Ситон - это шуточное название языка Сайтон по подобию с Пайтон/Питон, но есть и нешуточная часть: здесь будет теснее и больше к Си, поэтому оставляю статейку по [основам языка Си](https://thecode.media/osnovy-sintaksisa-yazyka-c/).

In [1]:
%load_ext cython

### Знание ограничений

Тип `char` - это простой тип, который как будто прост в понимании, правда:

- "char" происходит от "character" - символ, поэтому его можно называть символьным типом;
- правда символ - это целочисленный код, а значит `char` хранит целые (integer) числа;
- размер `char` - это байт, а байт - это минимально адресуемая область памяти, а не всегда 8 бит (смотри макрос **CHAR_BIT**);
- также нет ручательства (гарантии), что `char` - это всегда `signed char`, это может быть и `unsigned char`.

Довольно хитрый тип данных чтобы быть простецом.

In [2]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

# like `#include <limits.h>`
cimport libc.limits as limits

print(f"CHAR_BIT = {limits.CHAR_BIT}")

cdef signed char c = 'A'
cdef unsigned char uc = c  # safe implicit conversion
print(f"c = {c}; uc = {uc}")
print(f"c + 1 = {chr(c + 1)}; uc - 1 = {chr(uc - 1)}")

c = limits.CHAR_MAX
uc = limits.UCHAR_MAX
print(f"CHAR_MAX = {c}; UCHAR_MAX = {uc}")

# overflow(s)
c += 1   # UB = undefined behaviour
uc += 1  # DB = defined behaviour
print(f"Overflow(s): c={c} and uc={uc}")

CHAR_BIT = 8
c = 65; uc = 65
c + 1 = B; uc - 1 = @
CHAR_MAX = 127; UCHAR_MAX = 255
Overflow(s): c=-128 and uc=0


Вышний пример довольно показательный:

- удалось [использовать](https://cython.readthedocs.io/en/latest/src/tutorial/external.html) сишные макросы с помощью сайтоновского импорта (cimport);
- `char` - это целочисленный тип, а символ, например, `'A'` - это код 65 (см. [man ascii](https://man7.org/linux/man-pages/man7/ascii.7.html), а также за [типы](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#types) в Cython)
- воспроизведено переполнение типа: для знакового это неопределённое поведение (undefined behaviour), а вот для знакового делается круговой (циклический) сдвиг, или же "провёртка" (так я передаю забугорное "wrap-around").

Далее я предполагаю, что за целые и "плавающие" (float) числа знаю как я (надеюсь) и уважаемый читатель, поэтому оттаптываться на образцовых примерах вроде `0.1 + 0.2 != 0.3` я не буду, равно как и ссылаться на стандарт [IEEE 754](https://ru.wikipedia.org/wiki/IEEE_754) будто я сам его читал.

### Массивы

Массив - собрание (коллекция) для хранения однородных (гомогенных) элементов. Сайтон поддерживает два стиля объявления [массивов](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#c-arrays): Java и С. Легче и проще пользоваться "явовским" стилем, потому что он поддерживает объявление и "обначаливание" (инициализацию) в одну строку, тогда как для Си-стиля такое не прокатывает, считается непоощряемым и приводит к ошибке (на моей машине точно).

In [3]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

# like `from libc.math cimport sin`
cdef extern from "math.h":
    double sin(double x)


# C-style -> declare and initialise in one line
# cdef float point[3] = [0.1, -0.2, 0.3]  # Syntax error in C variable declaration

# C-style -> declare and then initialise separately -> possible yet discouraged
# cdef float point[3]
# point = [0.1, -0.2, 0.3]

# Java-style
cdef float[3] point = [0.1, -0.2, 0.3]
print(f"3D point -> {point}")

sines = [sin(point[i]) for i in range(len(point))]  # Python list
print(f"sines of the point -> {sines}")

3D point -> [0.10000000149011612, -0.20000000298023224, 0.30000001192092896]
sines of the point -> [0.0998334181294999, -0.19866933371588721, 0.29552021804983797]


Вместо сайтоновского импорта `from libc.math cimport sin` я решил попробовать "включить" (include) прототип функции `sin` из заголовочника (заголовочного файла - header file) [math.h](https://man7.org/linux/man-pages/man0/math.h.0p.html). Этот заголовочник стандартный, а значит он и его исходник (исходный файла - source file) известны системе, т.е. нет нужды прописывать пути к ним для компилятора.

Массив занимает некоторую область памяти, которая имеет начало и конец. Эта область памяти - ряд ячеек памяти (байтов), но в вышнем примере массив хранит "плавающие" (float) числа, а такое число может занимать, например, 4 байта. Получается, что массив содержит три "ячейки" (пока назову их так), и каждая ячейка массива состоит из 4х ячеек памяти (байтов), поэтому для массива из трёх "плавающих" чисел отводится 12 байт, т.е. "3 (число элементов) * 4 (размер типа данных)". Да что толковать, можно посмотреть:

In [4]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from cython cimport sizeof

cdef float[3] point = [0.1, -0.2, 0.3]

print(f"sizeof(float) = {sizeof(float)}")
print(f"sizeof(point) = {sizeof(point)}")
print(f"the length of point = {sizeof(point) // sizeof(float)}")

sizeof(float) = 4
sizeof(point) = 12
the length of point = 3


Видно, что в общем случае **"ячейка массива" не всегда то же, что и "ячейка памяти"**, хотя равенство возможно если это массив символов/байтов. Поэтому размер массива в памяти вычисляется по формуле `array_size * sizeof(item_type)`.

Я не зря веду речь за память и её области (regions) уже сейчас, потому что в Си работать с памятью приходится довольно часто, отсюда важно понимание её устройства и как с ней следует управляться. Именно поэтому "с сейчас" я начинаю ранний подступ к одному из мощнейших средств низкоуровневого программирования - указатели.

### Указатели

Указатели (pointers) - та тема, на которой Си указал мне на дверь, когда я пытался его учить самостоятельно. Когда я проходил бассейн в [Школе 21](https://21-school.ru/) (тогда ещё франшиза [École 42](https://42.fr/en/homepage/)), мне пришлось их освоить чтобы выжить и проплыть до победного завершения. В Пайтоне нет нужды в плотном соприкасании с памятью, а значит и указатели как средство языка убраны от греха подальше (а нагрешить легко). При этом мериловый (эталонный) Python написан как раз на языке Си (отсюда название его реализации [CPython](https://github.com/python/cpython)), а значит можно ожидать, что нутро CPython очень даже кишит указателями как [Фрай червями](https://ru.wikipedia.org/wiki/Parasites_Lost).

Простой (ну правда же) пример.

In [5]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stdint cimport uintptr_t

cdef unsigned char uc = 'W'
print(f"uc = {uc} ('{chr(uc)}')")

# ptr is a variable of type `pointer (*) to an unsigned char` 
cdef unsigned char *ptr
# now ptr stores the address (&) of the variable `uc`
ptr = &uc
print(f"address of uc (`&uc`) is {(<uintptr_t>ptr):#x}")

# using `*ptr` leads to error, `ptr[0]` saves the cell
print(f"value of `*ptr` is {ptr[0]}")

uc = 87 ('W')
address of uc (`&uc`) is 0x749fe4ab5390
value of `*ptr` is 87


Разбор примера

- объявлена переменная `unsigned char uc`, которой присвоен символ 'W' (код 87);
- эта переменная именует объект памяти - байт со значением 87 (ему хватит ячейки ибо байт);
- объявляю переменную `ptr`, тип которой - указатель/pointer (помечается `*`) на `unsigned char`, которой я присваиваю адрес (местоположение), переменной `uc` с помощью `&uc`
  -  `&uc` - это взятие адреса переменной `uc`, а не значения по ней
  -  **переменная-указатель хранит некий адрес** означает, что **эта переменная указывает на данные по этому адресу**
  -  с помощью оператора разыменования (dereferencing) можно получить доступ к данным по адресу, т.е. `*ptr` (здесь это также достижимо через `*(&uc)` - здесь звёздочка именно про доступ данных, а не про объявление указателя как это было раньше.
  - Сайтон оказался не без приколов: вместо сишного `*ptr` пришлось писать `ptr[0]`.

Но почему `ptr[0]`?! Это же не массив! Откуда индексация?! А всё из-за адресной арифметики, благодаря которой `ptr[0] = *(ptr + 0) = *ptr`. Здесь видна связь с массивами, которая раньше подслеповато обходилась стороной, и вот как так получается:

то неявный указатель на начало области памяти, которая ему отведена. Опишу на примере:

- Пусть `arr` - массив. Это имя - неявный указатель (звёздочки нет!) на начало отведённой ему области памяти.
- Доступ к первому элементу массива осуществим посредством `arr[0]` - элемент с нулевым смещением от его начала.
- Раз имени `arr` сопоставлен адрес области памяти, тогда `arr = arr + 0` является тем же адресом, но по нему можно взять данные с помощью разыменования, т.е. `*(arr + 0)`, что и выражается как `*arr` или `arr[0]`
- Но отсюда и следует, что `*ptr == *(ptr + 0) == ptr[0]`

Вот такое более тесное спорикасание массивов и указателей. Вот что можно написать на Си (примерное подобие):

```c
#include <stdio.h>

int main() {
	unsigned char uc = (unsigned char) 'W';  // type casting from `int`
	printf("uc = %hhu (%c)\n", uc, uc);

	unsigned char *ptr = &uc;
	printf("&uc (address of uc) = %p\n", ptr);
	printf("*ptr (fetch from the &uc) = %c\n", *ptr);

	return 0;
}
```

Пока хватит: указатели нужно постигать постепенно, дабы не отпугнуть всем-и-сразу.

### Игры с памятью

Если мне (скорее процессу) потребуется выделить дополнительную память, на выручку приходят две полезные функции:

- [malloc](https://man7.org/linux/man-pages/man3/malloc.3.html) - выделить (allocate) память процессу = затребовать у ОС
- [free](https://man7.org/linux/man-pages/man3/free.3p.html) - освободить ранее выделенную память - чтобы избежать [утечек памяти](https://coddy.tech/docs/ru/c/memory-leaks)

Благодаря им и происходит то, что по-умному называется [динамическим распределением памяти](https://ru.wikipedia.org/wiki/%D0%94%D0%B8%D0%BD%D0%B0%D0%BC%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D1%80%D0%B0%D1%81%D0%BF%D1%80%D0%B5%D0%B4%D0%B5%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BF%D0%B0%D0%BC%D1%8F%D1%82%D0%B8), или выделением во время исполнения, в динамике процесса. Для начала выделю-ка несколько байтиков для смехотворного примера.

In [6]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stddef cimport size_t
from libc.stdlib cimport malloc, free

cdef unsigned char n = 4
print(f"array length ({n}) * sizeof(char) ({sizeof(char)}) bytes to allocate")
# sizeof(char) is better than hardcoding it as 1
cdef char *s = <char *>malloc(n * sizeof(char))

# malloc may fail -> the best practice is to always check its return value
if s is NULL:
    raise MemoryError("malloc failed") from None

cdef size_t idx = 0
cdef char ch
# try to comment cdef for idx and ch and see the failure
for idx, ch in enumerate(b"abc"):
    s[idx] = ch
    idx += 1
print(f"idx = {idx} -> s[:idx] = {s[:idx]}")

# allocated memory must be free'd (returned to the system)
free(s)

array length (4) * sizeof(char) (1) bytes to allocate
idx = 3 -> s[:idx] = b'abc'


Самая мякотка здесь - это `cdef char *s = <char *>malloc(n * sizeof(char))` - выделить память (malloc = memory allocation = allocate memory) в размере `n * sizeof(char)` байт. Приведение возвращаемого типа (type casting) к типу `<char *>` избыточно, ибо переменная s уже имеет нужный тип и будет сделано неявное (implicit) приведение, а с другой стороны не мешает и поощримо, ведь **явное лучше неявного** (а также взаимность между C и C++). Возвраток (return value) от malloc нужно **обязательно проверять на NULL**, чтобы не разыменовать NULL-указатель и не поломать программу. И **обязательно зачищать/освобождать ранее выделенную память** когда та становится невостребованной.

Ранее было написано, что `char` может быть знаковым или беззнаковым (зависит от архитектуры), а в примере был использован `sizeof(char)`. Это видимая небрежность, ведь знаковость не влияет на размер самого типа. А ещё советую почитать вот этот вопрос на [StackOverflow](https://stackoverflow.com/questions/69810915/why-is-sizeof-a-char-literal-not-the-same-as-sizeofchar) про взятие размера типа по объекту/литералу.

Вышний пример обнажает ещё одну прелесть: возможность использовать срезы (slices). Здесь срез понадобился для безопасной пропечати данных из массива во избежание неопределённое поведения с вероятным падением. А дело, что **массив в Си - это "голая" область памяти, а не структура данных, которая знает свой размер**, а значит не понятно сколько именно байтов прочитать по имеющемуся адресу. Однако в случае массива символов можно обойтись и без явного знания его длины, есть особая помета (марка), которая позволит определять где заканчивается подобный массив символов, который таким образом становится строкой. 

### Строки

В Пайтоне (я не рассматриваю версии меньше Python3) есть тип строка [str](https://docs.python.org/3/builtins/stdtypes.html#text-sequence-type-str) - мощные, [юникодные](https://home.unicode.org/) и сверхполноценные. Также есть тип [bytes](https://docs.python.org/3/builtins/stdtypes.html#bytes) - байтовые последовательности, или байтовые строки. А вот **в языке Си строка как отдельный тип данных отсутствует**! Никакого противоречия: когда говорят про строку, на самом деле говорят за **массив символов с последним элементом '\0'**. Символ '\0' помечает конец строки, поэтому он называется нуль-терминатором (NUL-terminator, сиречь нуль-завершитель), который [не то же самое, что NULL](https://stackoverflow.com/questions/1296843/what-is-the-difference-between-null-0-and-0).

Строки как символы: кажутся простыми, а на деле...

In [7]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stddef cimport size_t
from libc.stdlib cimport malloc, free
from libc.string cimport strcpy, strcat, strlen

# byte strings are closer to ASCII C strings
cdef const char* s1 = b"Hell\0o"  # not "Hello", but "Hell" because of '\0'
cdef const char* s2 = b"World"   # '\0' is not required by default

cdef size_t n = strlen(s1) + strlen(s2) + 1  # +1 for '\0'
cdef char* s3 = <char*>malloc(n)

# malloc can fail, therefore it is vital to check the result
if s3 == NULL:
    raise MemoryError("malloc failed") from None

try:
    strcpy(s3, s1)  # writes s1 in the empty buffer s3
    strcat(s3, s2)  # (con)catenates s2 to the end of s1
    print(s3)
finally:
    free(s3)

b'HellWorld'


Использование байтовых строк необходимо для явного обеспечения [ASCII](https://ru.wikipedia.org/wiki/ASCII) режима во избежания международного скандала с участием Юникода. Строки **s1** и **s2** особенны тем, что их типы помечены [квалификатором](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#type-qualifiers) const - так я хочу заявить, что данные "строки" не должны изменяться (хотя это можно обойти, так не стоит делать). А далее простой алгоритм:

- выделение памяти под новую строку (совокупная длина двух строк) с учётом '\0' знака/символа (+1 байт);
- копирование строки s1 в буфер s3, т.е. область памяти, на которую ссылается переменная-указатель s3 - [strcpy](https://man7.org/linux/man-pages/man3/strcpy.3.html) добавил завершающий нуль-символ;
- сцепление (конкатенация) строки s2 с уже строкой s3 - уже [strcat](https://man7.org/linux/man-pages/man7/strcat.7.html) добавил нуль=завершитель от себя.

В этом примере кроется вообще-то неприличный недостаток: нет проверки на переполнение типов. Строки могут быть столь длинными, что их сложение или добавка всего одного байта может не влезть в size_t, а значит на самом деле выделенный объём памяти будет неприравненным (неадэкватным) и это уже ошибка. По-хорошему, такую проверку нужно сделать, но сие избыточно для этого примера и его поучительная часть такая: **не нужно усложнять там где это не нужно**. Ну видно же, что длины строк достаточно уместительные, можно и попроще.

### Итоги

Довольно утешительные, удалось показать как будто достаточно для сего раза:

- многоликость `char`
- одномерные массивы
- указатели и робкий показ их взаимосвязи с массивами
- динамическое выделение памяти - связка malloc (и ребята) с free
- строки как нуль-терминированный массив символов (линейная область ячеек памяти)
- некоторые сишные функции и как их вызывать да пользовать в Сайтоне

Пора останавливаться, что и делаю.